# DGHV com decifração esmagada

Implementação da transformação da Secção 6.1 do paper: a chave secreta passa a ser um vetor esparso `s`, a chave pública recebe valores `y_i = u_i / 2^k`, e cada texto cifrado fica com a forma `(c_star, z)`.

A decifração segue a fórmula do paper:

```text
m = [c_star - round(sum(s_i * z_i))]_2
```

Depois, `bootstrap` avalia homomorficamente esta decifração usando a chave de bootstrapping `Enc(s_i)`. Ou seja, o refresh não chama `decifrar` seguido de `cifrar`.

Nota: os parâmetros do teste são toy para o notebook correr. A construção segue o mecanismo do paper, mas não tem segurança prática.

In [ ]:
from secrets import randbelow

from somewhat_homomorphic import *

In [ ]:
def ceil_log2(value):
    return max(0, (value - 1).bit_length())


def arredondar_divisao(numerador, denominador):
    if denominador <= 0:
        raise ValueError("denominador deve ser positivo")

    if numerador >= 0:
        return (numerador + denominador // 2) // denominador

    return -((-numerador + denominador // 2) // denominador)

class CifraEsmagada:
    def __init__(self, c_star, zs):
        self.c_star = c_star
        self.zs = zs
        self.value = c_star

    def __str__(self):
        return f"CifraEsmagada(c_star={self.c_star}, zs=<len {len(self.zs)}>)"


class DGHVEsmagado:
    def __init__(self, params=None, k=None, theta=8, big_theta=128):
        if theta <= 0 or big_theta < theta:
            raise ValueError("0 < theta <= big_theta")

        self.params = params or Parametros()
        self.base = DGHV(self.params)
        # k = gamma*eta/rho'
        self.k = k or max(
            self.params.gamma + 8,
            (self.params.gamma * self.params.eta) // max(1, self.params.rho_prime),
        )
        self.theta = theta
        self.big_theta = big_theta
        # n = ceil(log(theta)) + 3
        self.n = ceil_log2(theta) + 3
        # escala_z = 2^n,
        self.escala_z = 1 << self.n
        self.s, self.us = self.gerar_key_esmagada()

    def gerar_key_esmagada(self):
        p = self.base.keys.sk.p
        # u_i em [0, 2^(k+1)); modulo = 2^(k+1)
        modulo = 1 << (self.k + 1)
        # x_p = round(2^k/p) 
        x_p = arredondar_divisao(1 << self.k, p)

        posicoes = []
        usadas = set()
        while len(posicoes) < self.theta:
            pos = randbelow(self.big_theta)
            if pos not in usadas:
                usadas.add(pos)
                posicoes.append(pos)

        # s=(s_1,...,s_Theta), sum_i s_i = theta
        s = [1 if i in usadas else 0 for i in range(self.big_theta)]
        
        us = [randbelow(modulo) for _ in range(self.big_theta)]

        # sum_i s_i*u_i = x_p mod 2^(k+1)
        ultima = posicoes[-1]
        soma_parcial = sum(us[i] for i in posicoes[:-1])
        us[ultima] = (x_p - soma_parcial) % modulo

        return s, us

    def calcular_vetor_zs(self, c_star):
        modulo = 1 << (self.k + 1)
        zs = []

        for u in self.us:
            # y_i = u_i/2^k e z_i = [c_star*y_i]_2 
            residuo = (c_star * u) % modulo
            z = arredondar_divisao(residuo * self.escala_z, 1 << self.k)
            zs.append(z % (2 * self.escala_z))

        return zs

    def cria_cifra_esmagada(self, c_star):
        return CifraEsmagada(c_star, self.calcular_vetor_zs(c_star))

    def _enc(self, bit):
        return self.cria_cifra_esmagada(self.base.enc(bit).value)

    def _dec(self, cifra):
        # m' = [c_star - round(sum_i s_i*z_i)]_2
        total = sum(si * zi for si, zi in zip(self.s, cifra.zs))
        quociente_aproximado = arredondar_divisao(total, self.escala_z)
        return (cifra.c_star - quociente_aproximado) % 2

    def const(self, bit):
        return self.cria_cifra_esmagada(bit & 1)

    def xor(self, esquerda, direita):
        return self.soma(esquerda, direita)

    def and_(self, esquerda, direita):
        return self.multiplicar(esquerda, direita)

    def gerar_chave_bootstrap(self):
        # chave_bootstrap = (Enc(s_1),...,Enc(s_Theta))
        self.chave_bootstrap = [self.enc(bit) for bit in self.s]
        return self.chave_bootstrap

    def bits_peso_coluna(self, entradas):
        # bit j do peso de Hamming = e_(2^j)(sigma) mod 2
        num_bits = ceil_log2(self.theta + 1)
        max_grau = 1 << (num_bits - 1)
        pols = [self.const(0) for _ in range(max_grau + 1)]
        pols[0] = self.const(1)

        for sigma in entradas:
            for grau in range(max_grau, 0, -1):
                # e_g <- e_g + sigma*e_(g-1) mod 2 
                pols[grau] = self.xor(pols[grau], self.and_(sigma, pols[grau - 1]))

        return [pols[1 << bit] for bit in range(num_bits)]

    def somar_bits(self, esquerda, direita):
        # soma = a xor b xor carry; carry'=(a and b) xor (carry and (a xor b)) 
        carry = self.const(0)
        resultado = []

        for a, b in zip(esquerda, direita):
            a_xor_b = self.xor(a, b)
            resultado.append(self.xor(a_xor_b, carry))
            carry = self.xor(self.and_(a, b), self.and_(carry, a_xor_b))

        resultado.append(carry)
        return resultado

    def bootstrap(self, cifra):
        # if not self.chave_bootstrap:
        #     self.gerar_chave_bootstrap()

        bits_contador = ceil_log2(self.theta + 1)
        largura = self.n + bits_contador + 2
        # somar 2^(n-1) antes de extrair o bit n implementa arredondamento por 2^n 
        arredondamento = 1 << (self.n - 1)
        total = [self.const((arredondamento >> j) & 1) for j in range(largura)]
        zero = self.const(0)

        for coluna in range(self.n + 1):
            # a_i = s_i*z_i
            entradas = [
                enc_s if ((z >> coluna) & 1) else zero
                for enc_s, z in zip(self.chave_bootstrap, cifra.zs)
            ]
            peso = self.bits_peso_coluna(entradas)
            termo = [zero for _ in range(largura)]

            for bit, enc_bit in enumerate(peso):
                if coluna + bit < largura:
                    termo[coluna + bit] = enc_bit

            total = self.somar_bits(total, termo)[:largura]

        # Enc(round(sum_i s_i*z_i) mod 2) xor (c_star mod 2) 
        bit_quociente = total[self.n]
        return self.xor(bit_quociente, self.const(cifra.c_star & 1))

    # def evaluate(self, circuit, ciphertexts):
    #     stack = []
    #     for token in circuit:
    #         # if token in ciphertexts:
    #         #     print(ciphertexts[token].value)
    #         if token == "ADD":
    #             right = stack.pop()
    #             left = stack.pop()
    #             stack.append(self.soma(left, right))
    #         elif token == "MULT":
    #             right = stack.pop()
    #             left = stack.pop()
    #             stack.append(self.multplicacao(left, right))
    #         else:
    #             stack.append(ciphertexts[token])
    #     if len(stack) != 1:
    #         raise ValueError("circuito invalido")
    #     return stack[0]

    def avaliar_bootstrap(self, circuito, cifras):
        stack = []

        for token in circuito:
            if token == "ADD":
                direita = stack.pop()
                esquerda = stack.pop()
                stack.append(self.bootstrap(self.soma(esquerda, direita)))
            elif token == "MULT":
                direita = stack.pop()
                esquerda = stack.pop()
                stack.append(self.bootstrap(self.multiplicar(esquerda, direita)))
            else:
                stack.append(cifras[token])

        if len(stack) != 1:
            raise ValueError("circuito inválido")

        return stack[0]

    def soma(self, esquerda, direita):
        # DecSq(P(c1*+c2*)) = m1 xor m2 
        return self.cria_cifra_esmagada(esquerda.c_star + direita.c_star)

    def multiplicar(self, esquerda, direita):
        # DecSq(P(c1*c2)) = m1 and m2 
        return self.cria_cifra_esmagada(esquerda.c_star * direita.c_star)

    def decifrar_original_para_teste(self, cifra):
        return self.base.dec(Ciphertext(cifra.c_star))

    def condicoes_do_paper(self, cifra):
        # |c_star| < 2^(k-4) e ruido(c_star) < p/8 
        p = self.base.keys.sk.p
        ruido = abs(mod_perto_0(cifra.c_star, p))
        return {
            "|c_star| < 2^(k-4)": abs(cifra.c_star) < (1 << (self.k - 4)),
            "ruido < p/8": 8 * ruido < p,
        }

    # enc = _enc
    # dec = _dec
    # # evaluate = avaliar
    
    def __str__(self):
        return (
            f"DGHVEsmagado(k={self.k}, theta={self.theta}, "
            f"big_theta={self.big_theta}, n={self.n})"
        )

In [3]:
params = Parametros(eta=160, gamma=640, rho=8, rho_prime=12, tau=32)
Cif_2 = DGHVEsmagado(params, theta=2, big_theta=4)
Cif_2.gerar_chave_bootstrap()

bits_2 = {
    "a": 1,
    "b": 1,
    "c": 1,
    "d": 1,
    "e": 1,
    "f": 1,
    "g": 1,
    "h": 1,
    "i": 1,
    "j": 1,
    "k": 1,
    "l": 1,
}

circuito_2 = [
    "a", "b", "MULT",
    "c", "MULT",
    "d", "MULT",
    "e", "MULT",
    "f", "MULT",
    "g", "MULT",
    "h", "MULT",
    "i", "MULT",
    "j", "MULT",
    "k", "MULT",
    "l", "MULT",
]

cifras_2 = {nome: Cif_2.enc(bit) for nome, bit in bits_2.items()}
resultado_2 = Cif_2.avaliar_bootstrap(circuito_2, cifras_2)
esperado_2 = 1
obtido_2 = Cif_2._dec(resultado_2)

print(Cif_2)
print("peso da chave secreta s:", sum(Cif_2.s))
print("tamanho da chave de bootstrapping:", len(Cif_2.chave_bootstrap))
print("resultado esperado:", esperado_2)
print("resultado com bootstrapping:", obtido_2)
print("condições finais do paper:", Cif_2.condicoes_do_paper(resultado_2))

assert obtido_2 == esperado_2

DGHVEsmagado(k=8533, theta=2, big_theta=4, n=4)
peso da chave secreta s: 2
tamanho da chave de bootstrapping: 4
resultado esperado: 1
resultado com bootstrapping: 1
condições finais do paper: {'|c_star| < 2^(k-4)': True, 'ruido < p/8': True}
